# Losses Notebook

- Source: `src/losses.py`
- 목적: 원본 파이썬 파일을 단계별로 실행/설명하기 위한 노트북 버전
- 실행 방법: 위에서 아래로 순서대로 실행


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    # 노트북이 다른 경로에서 열렸을 때 프로젝트 루트 자동 탐색
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / 'src').exists() and (parent / 'configs').exists():
            PROJECT_ROOT = parent
            break
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')


## Step 1. Setup and Imports

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
"""Loss functions.

CrossEntropyLoss (기본)
- label_smoothing : 모델이 정답에 너무 자신만만해지는 것을 막는다.
  예) smoothing=0.1 → 정답 클래스 타깃이 1.0 대신 0.9로, 나머지가 0.1/C씩
- class weight    : 클래스 불균형 보정. compute_class_weights() 결과를 바로 사용.

FocalLoss (선택)
- 쉬운 샘플(이미 잘 맞추는 것)의 loss를 줄이고 어려운 샘플에 집중한다.
- gamma=2.0이 일반적인 기본값.
- 결점두처럼 클래스 불균형이 매우 심할 때 CE보다 효과적일 수 있다.
  (현재는 train.py가 CE만 사용하지만, FocalLoss로 교체도 가능)
"""
from __future__ import annotations
import torch
import torch.nn as nn
import torch.nn.functional as F


## Step 2. Function: cross_entropy_loss

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
def cross_entropy_loss(weight: torch.Tensor | None = None,
                       label_smoothing: float = 0.0) -> nn.Module:
    # 현재 프로젝트의 기본 loss.
    # class weight로 불균형을 보정하고, label smoothing으로 과신을 줄인다.
    return nn.CrossEntropyLoss(weight=weight, label_smoothing=label_smoothing)


## Step 3. Class: FocalLoss

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
class FocalLoss(nn.Module):
    """클래스 불균형이 매우 심할 때만 사용."""


## Step 4. Function: __init__

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
    def __init__(self, alpha: torch.Tensor | None = None, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma


## Step 5. Function: forward

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        # 쉬운 샘플의 loss는 줄이고, 어려운 샘플에 더 집중하게 만드는 형태다.
        ce = F.cross_entropy(logits, target, weight=self.alpha, reduction="none")
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()
